# 2 · Pairwise Alignment of Amino Acid Sequences
Computes all-vs-all pairwise percent sequence identity, saves results, then filters by a similarity threshold and builds a partner lookup dictionary.

> **Note:** `Bio.pairwise2` was removed in Biopython 1.84. This notebook uses `Bio.Align.PairwiseAligner`.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
FASTA_FILE          = "./metadata/empiar_seq.fasta"
NPY_OUT             = "./metadata/pairwise_alignment_empiar.npy"
PERCENTAGE_SIMILARITY = 30   # minimum % identity to keep a pair
N_WORKERS           = 0      # 0 = use all available CPUs

In [2]:
import numpy as np
from itertools import combinations
from multiprocessing import Pool, cpu_count
from Bio import SeqIO
from Bio.Align import PairwiseAligner

_aligner = PairwiseAligner()
_aligner.mode           = "global"
_aligner.match_score    = 1
_aligner.mismatch_score = 0
_aligner.open_gap_score = 0
_aligner.extend_gap_score = 0

## 1 · Compute pairwise identity

In [3]:
def percent_identity(args):
    """
    Return (id1, id2, %identity) or None if the pair shares the same
    5-character ID prefix (i.e. same EMPIAR dataset).
    """
    rec1, rec2 = args
    if rec1.id[:5] == rec2.id[:5]:
        return None
    top        = next(iter(_aligner.align(str(rec1.seq), str(rec2.seq))))
    aligned_a, aligned_b = top[0], top[1]
    matches    = sum(a == b for a, b in zip(aligned_a, aligned_b))
    pid        = 100.0 * matches / len(aligned_a)
    return rec1.id, rec2.id, pid

In [4]:
sequences      = list(SeqIO.parse(FASTA_FILE, "fasta"))
sequence_pairs = list(combinations(sequences, 2))
workers        = N_WORKERS if N_WORKERS > 0 else cpu_count()
print(f"Computing {len(sequence_pairs):,} pairs using {workers} workers …")

with Pool(processes=workers) as pool:
    results = pool.map(percent_identity, sequence_pairs)

pairwise_alignment = np.array([r for r in results if r is not None], dtype=object)
print(f"Non-skipped pairs: {len(pairwise_alignment):,}")
np.save(NPY_OUT, pairwise_alignment)
print(f"Saved → {NPY_OUT}")

Computing 23,205,078 pairs using 104 workers …


## 2 · Filter by similarity threshold

In [5]:
pairwise_alignment = np.load(NPY_OUT, allow_pickle=True)

# Extract all EMPIAR IDs present in the alignment
empiar_ids = np.unique([
    id_str
    for entry in pairwise_alignment
    for id_str in [entry[0].split("_")[0], entry[1].split("_")[0]]
])
print(f"Unique EMPIAR IDs in alignment: {len(empiar_ids)}")

pairwise_alignment = pairwise_alignment[
    pairwise_alignment[:, -1].astype(float) > PERCENTAGE_SIMILARITY
]
print(f"Pairs above {PERCENTAGE_SIMILARITY}%: {len(pairwise_alignment):,}")

Unique EMPIAR IDs in alignment: 343
Pairs above 30%: 46,389


## 3 · Build unique match lookup dictionary

In [6]:
unique_matches = []

for pairs in pairwise_alignment:
    pair_1, pair_2, perc = pairs
    empiar_p1, emd_p1 = pair_1.split("_")[:2]
    empiar_p2, emd_p2 = pair_2.split("_")[:2]

    umta = np.array(unique_matches) if unique_matches else np.empty((0, 5))
    fm = np.where((umta[:, 0] == empiar_p1) & (umta[:, 2] == empiar_p2))[0]
    sm = np.where((umta[:, 0] == empiar_p2) & (umta[:, 2] == empiar_p1))[0]

    if len(fm):
        if perc > unique_matches[int(fm[0])][-1]:
            unique_matches[int(fm[0])][-1] = perc
        continue
    if len(sm):
        if perc > unique_matches[int(sm[0])][-1]:
            unique_matches[int(sm[0])][-1] = perc
        continue
    unique_matches.append([empiar_p1, emd_p1, empiar_p2, emd_p2, perc])

unique_matches = np.array(unique_matches)

seen_pairs = set()
unique_matches_dict = {}

for ent in unique_matches:
    empiar1, emd1, empiar2, emd2, perc = ent

    # Canonical key: sort the two sides so (A,B) and (B,A) hash the same
    pair_key = tuple(sorted([(empiar1, emd1), (empiar2, emd2)]))
    if pair_key in seen_pairs:
        continue
    seen_pairs.add(pair_key)

    unique_matches_dict.setdefault(empiar1, []).append([emd1, empiar2, emd2, perc])

unique_matches_dict = {k: np.array(v) for k, v in unique_matches_dict.items()}
print(f"Unique match pairs: {len(unique_matches_dict)}")

Unique match pairs: 204


## 4 · Entries with 4+ partners

In [7]:
unique_matches_dict_multiple = {k: v for k, v in unique_matches_dict.items() if len(v) >= 4}
print(f"Entries with 4+ partners: {len(unique_matches_dict_multiple)}")

Entries with 4+ partners: 104


In [8]:
# ---------------------------------------------------------------------------
# Static selected EMPIAR ID lists
# ---------------------------------------------------------------------------

micrograph_entries = (
    '10004', '10005', '10012', '10017', '10025', '10056', '10061', '10075', '10122', '10160',
    '10175', '10189', '10190', '10192', '10193', '10199', '10203', '10208', '10217', '10240',
    '10268', '10271', '10283', '10289', '10290', '10291', '10379', '10401', '10411', '10425',
    '10433', '10467', '10475', '10489', '10519', '10560', '10590', '10652', '10667', '10706',
    '10707', '10735', '10760', '10790', '10794', '10800', '10882', '10888', '10890', '10891',
    '10893', '10919', '10920', '10930', '11060', '11131', '11139', '11146', '11149', '11167',
    '11193', '11194', '11204', '11267', '11268', '11341', '11374', '11443', '11501', '11524',
    '11553', '11556', '11567', '11604', '11605', '11607', '11608', '11615', '11703', '11719',
    '11726', '11755', '11761', '11763', '11768', '11789', '11795', '11840', '11847', '11954',
    '11986', '12087', '12106', '12140', '12143', '12171', '12237', '12240', '12310', '12311',
    '12486', '12561', '12562', '12667',
)

particle_entries = (
    '10024', '10028', '10044', '10049', '10059', '10063', '10072', '10076', '10081', '10090',
    '10091', '10093', '10096', '10097', '10099', '10107', '10123', '10124', '10127', '10128',
    '10166', '10176', '10180', '10202', '10229', '10254', '10264', '10278', '10279', '10285',
    '10294', '10299', '10307', '10308', '10309', '10317', '10328', '10330', '10333', '10335',
    '10336', '10341', '10342', '10344', '10345', '10347', '10350', '10357', '10358', '10373',
    '10374', '10380', '10391', '10395', '10396', '10397', '10398', '10399', '10406', '10407',
    '10409', '10420', '10421', '10437', '10443', '10454', '10455', '10465', '10470', '10481',
    '10482', '10483', '10532', '10536', '10640', '10659', '10669', '10697', '10703', '10722',
    '10739', '10752', '10770', '10786', '10792', '10810', '10841', '10873', '10874', '10876',
    '11000', '11005', '11043', '11120', '11128', '11211', '11233', '11247', '11270', '11283',
    '11362', '11521', '11522', '11523', '11526', '11618', '11665', '11681', '11706', '11720',
    '11734', '11762', '11791', '11792', '11796', '11797', '11834', '11836', '11844', '11898',
    '11910', '11925', '11987', '12036', '12093', '12094', '12097', '12121', '12180', '12443',
    '12444', '12510',
)

total_entries = micrograph_entries + particle_entries

# This line is a no-op in a script but useful when run interactively.
len(particle_entries), len(micrograph_entries), len(total_entries)

(132, 104, 236)